# MLOps Class 1: Churn prediction, end to end
NumPy → pandas → scikit-learn → save → serve → monitor drift.

**Run all** from the Runtime menu in Colab, or step through cell by cell.

# Demo 1: data to a trained model
Cells 0-4: NumPy → pandas → scikit-learn pipeline → train and evaluate.

## 0. Setup
All libraries are preinstalled in Colab. Locally: `pip install numpy pandas scikit-learn scipy joblib`.

In [ ]:
import numpy as np, pandas as pd, joblib
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

## 1. Data (NumPy)
- Seeded generator = **reproducible** data (first MLOps idea).
- Messy on purpose: NaNs and duplicates.

**Ask:** without a seed, can I compare tomorrow's results with today's?

In [ ]:
def make_data(n, seed, login_shift=0):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "tenure_months":   rng.integers(1, 72, n),
        "monthly_charges": rng.normal(70, 20, n).clip(20, 150).round(2),
        "logins_per_week": rng.poisson(5 + login_shift, n),
        "support_tickets": rng.poisson(1.5, n),
        "plan":            rng.choice(["basic", "standard", "premium"], n, p=[.5, .3, .2]),
    })
    logit = (-0.5 - 0.04*df.tenure_months + 0.015*df.monthly_charges
             - 0.25*df.logins_per_week + 0.5*df.support_tickets + 0.6*(df.plan == "basic"))
    df["churn"] = (rng.random(n) < 1/(1 + np.exp(-logit))).astype(int)
    df.loc[rng.choice(n, int(.03*n), replace=False), "monthly_charges"] = np.nan
    return df

df = make_data(3000, seed=42)
df = pd.concat([df, df.sample(25, random_state=1)], ignore_index=True)   # dirty: duplicates

## 2. Explore (pandas)
- Look at the data before modelling: missing values, duplicates, class balance, `groupby`.

**Ask:** basic plans churn most. What would you do with that?

In [ ]:
print(df.shape, "| churn rate:", df.churn.mean().round(3))
print(df.isna().sum()[lambda s: s > 0])
print("duplicates:", df.duplicated().sum())
print(df.groupby("plan").churn.mean().round(3))
print(df.describe().round(1))

## 3. Prep and Pipeline
- One `add_features()` used for training **and** serving avoids training-serving skew.
- Pipeline bundles prep + model so nothing is forgotten at serving time.

**Ask:** what goes wrong if we scale before the train/test split? (data leakage)

In [ ]:
def add_features(d):
    d = d.copy()
    d["tickets_per_month"] = d.support_tickets / d.tenure_months
    return d

df = df.drop_duplicates()
X, y = add_features(df.drop(columns="churn")), df.churn
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

num = ["tenure_months", "monthly_charges", "logins_per_week", "support_tickets", "tickets_per_month"]
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["plan"]),
])

## 4. Train and evaluate
- `stratify` keeps the churn rate equal in train/test.
- AUC = general quality; recall in top 10% = business metric.
- Simple model can beat a fancier one.

**Ask:** would you ship AUC 0.79? What else do you need to know?

In [ ]:
results = {}
for name, model in {"logreg": LogisticRegression(max_iter=1000),
                    "gboost": GradientBoostingClassifier(random_state=42)}.items():
    pipe = Pipeline([("pre", pre), ("model", model)]).fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    top10 = np.argsort(p)[::-1][: len(p) // 10]
    results[name] = (pipe, roc_auc_score(y_test, p), y_test.to_numpy()[top10].sum() / y_test.sum())
    print(f"{name:7s} AUC={results[name][1]:.3f}  recall@top10%={results[name][2]:.2f}")

best = max(results, key=lambda k: results[k][1])
print("best:", best)

# Demo 2: from a model to MLOps
Cells 5-7: save → serve → monitor drift → track. Continue in the same session; these cells reuse the model from Demo 1.

## 5. Save and serve
- `joblib.dump` = model artifact, the training to deployment handoff.
- `predict_churn()` is a tiny prediction API; version the file (`v1`) for rollback.

**Ask:** how would you wrap this as a web service?

In [ ]:
joblib.dump(results[best][0], "churn_model_v1.joblib")

def predict_churn(customer: dict) -> float:
    model = joblib.load("churn_model_v1.joblib")           # in production: load once at startup
    return float(model.predict_proba(add_features(pd.DataFrame([customer])))[0, 1])

print("risky  :", round(predict_churn({"tenure_months": 3, "monthly_charges": 95, "logins_per_week": 1, "support_tickets": 4, "plan": "basic"}), 3))
print("loyal  :", round(predict_churn({"tenure_months": 60, "monthly_charges": 60, "logins_per_week": 9, "support_tickets": 0, "plan": "premium"}), 3))

## 6. Monitor: data drift
- Accurate today does not mean accurate next month.
- KS test compares live vs training data; `logins_per_week` is flagged because we shifted it.

**Ask:** drift detected: auto-retrain or human decision?

In [ ]:
live = make_data(1000, seed=7, login_shift=-2)             # simulate: customers logging in less
for col in ["tenure_months", "monthly_charges", "logins_per_week", "support_tickets"]:
    stat, pval = ks_2samp(df[col].dropna(), live[col].dropna())
    print(f"{col:16s} p={pval:.4f}", "<-- DRIFT: investigate / retrain" if pval < 0.01 else "ok")

## 7. (Optional) Experiment tracking
MLflow preview for a later class. Run `!pip install mlflow` first, or skip this cell.

In [ ]:
try:
    import mlflow
    with mlflow.start_run(run_name="churn_v1"):
        mlflow.log_param("model", best)
        mlflow.log_metric("auc", results[best][1])
        mlflow.log_metric("recall_top10", results[best][2])
    print("logged to ./mlruns")
except ImportError:
    print("mlflow not installed, skipping")

## Takeaway
Everything above ran manually, once. **MLOps automates, versions, tests and monitors each step.**